#### Implement a unified gold-standard training dataset that can be used for the final fine-tuning and testing after the silver training run.

In [126]:
# Imports + label mappings
import os
import json
from pathlib import Path
import pandas as pd
import psycopg2
from dotenv import load_dotenv
import torch
import psycopg2
import pandas as pd

load_dotenv()

# Verify GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")




Device: cuda
GPU: NVIDIA GeForce RTX 4070 Ti SUPER
VRAM: 17.2 GB


In [127]:
# Standard 15-frame labels (aligns with earlier project)
OFFICIAL_LABELS = [
    'Economic',
    'Capacity and resources',
    'Morality',
    'Fairness and equality',
    'Legality, constitutionality and jurisprudence',
    'Policy prescription and evaluation',
    'Crime and punishment',
    'Security and defense',
    'Health and safety',
    'Quality of life',
    'Cultural identity',
    'Public opinion',
    'Political',
    'External regulation and reputation',
    'Other',
]

# MFC frame code mapping
MFC_CODE_TO_LABEL = {
    1: 'Economic',
    2: 'Capacity and resources',
    3: 'Morality',
    4: 'Fairness and equality',
    5: 'Legality, constitutionality and jurisprudence',
    6: 'Policy prescription and evaluation',
    7: 'Crime and punishment',
    8: 'Security and defense',
    9: 'Health and safety',
    10: 'Quality of life',
    11: 'Cultural identity',
    12: 'Public opinion',
    13: 'Political',
    14: 'External regulation and reputation',
    15: 'Other',
}

# SemEval frame name -> standard label mapping
SEMEVAL_TO_LABEL = {
    'Economic': 'Economic',
    'Capacity_and_resources': 'Capacity and resources',
    'Morality': 'Morality',
    'Fairness_and_equality': 'Fairness and equality',
    'Legality_Constitutionality_and_jurisprudence': 'Legality, constitutionality and jurisprudence',
    'Policy_prescription_and_evaluation': 'Policy prescription and evaluation',
    'Crime_and_punishment': 'Crime and punishment',
    'Security_and_defense': 'Security and defense',
    'Health_and_safety': 'Health and safety',
    'Quality_of_life': 'Quality of life',
    'Cultural_identity': 'Cultural identity',
    'Public_opinion': 'Public opinion',
    'Political': 'Political',
    'External_regulation_and_reputation': 'External regulation and reputation',
}

def mfc_union_labels(frame_annotations):
    """Union all frame codes across annotators -> label list."""
    if frame_annotations is None:
        return []
    # psycopg2 may return dict; sometimes a JSON string
    if isinstance(frame_annotations, str):
        try:
            frame_annotations = json.loads(frame_annotations)
        except json.JSONDecodeError:
            return []
    if not isinstance(frame_annotations, dict):
        return []

    all_codes = set()
    for ann_list in frame_annotations.values():
        if not ann_list:
            continue
        for ann in ann_list:
            code = ann.get('code') if isinstance(ann, dict) else None
            if code is None:
                continue
            base_code = int(float(code))
            if base_code in MFC_CODE_TO_LABEL:
                all_codes.add(base_code)
    return [MFC_CODE_TO_LABEL[c] for c in sorted(all_codes)]


In [128]:
conn = psycopg2.connect(
    dbname=os.getenv('DB_NAME'),
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD'),
    host=os.getenv('DB_HOST'),
    port=os.getenv('DB_PORT'),
)
cur = conn.cursor()
cur.execute("""
    SELECT id, article_id, title, topic, text_cleaned, frame_annotations
    FROM media_frames_corpus
    WHERE frame_annotations IS NOT NULL AND frame_annotations <> '{}'::jsonb
""")
rows = cur.fetchall()
cur.close()
conn.close()

mfc = pd.DataFrame(rows, columns=["id", "article_id", "title", "topic", "text_cleaned", "frame_annotations"])
print(f"Loaded {len(mfc):,} articles")
mfc.head()

Loaded 2,224 articles


,id,article_id,title,topic,text_cleaned,frame_annotations
0,486,Immigration1.0-1371,THE FINE PRINT: A close look at the immigratio...,immigration,"By nearly all accounts, Karen Zacarias, a 26-y...","{'annotator0': [5, 6, 10, 11], 'annotator4': [..."
1,975,Immigration1.0-20547,Press One for English,immigration,The immigration debate in Congress has hit sev...,"{'annotator3': [4, 11, 13], 'annotator6': [3, ..."
2,976,Immigration1.0-17350,Prosecutors Say Defendant in Immigrant Smuggli...,immigration,The Chinese businesswoman called Sister Ping w...,"{'annotator1': [1, 7, 9, 11], 'annotator10': [..."
3,977,Immigration1.0-19222,Protests Go On In Several Cities As Panel Acts,immigration,Tens of thousands of immigrants here and in se...,"{'annotator8': [5, 7, 8, 11, 12], 'annotator9'..."
4,978,Immigration1.0-18843,Refining The Tests That Confer Citizenship,immigration,Where does Father Christmas come from? How old...,"{'annotator3': [1, 5, 6], 'annotator7': [6, 15]}"


In [129]:
# Clean the annotations and apply keys
# mfc['present_frames'] = 

output = []
for item_dict in list(mfc['frame_annotations']):
    to_add = list(item_dict.values())
    output.append(to_add)

new_output = [annotations for row in output for annotations in row]
new_output = [
    [x for sublist in row for x in sublist]
    for row in output
]
new_output = [list(set(sublist)) for sublist in new_output]
mfc['present_frames_keys'] = new_output


def convert_list_to_labels(integer_list: list):
    """converts list of integer labels to list of descriptive labels"""
    desc_list = [MFC_CODE_TO_LABEL[integer] for integer in integer_list]
    return desc_list
    
mfc['present_frames'] = mfc['present_frames_keys'].apply(convert_list_to_labels)

mfc['source'] = 'mfc'

mfc.head()


,id,article_id,title,topic,text_cleaned,frame_annotations,present_frames_keys,present_frames,source
0,486,Immigration1.0-1371,THE FINE PRINT: A close look at the immigratio...,immigration,"By nearly all accounts, Karen Zacarias, a 26-y...","{'annotator0': [5, 6, 10, 11], 'annotator4': [...","[5, 6, 10, 11, 12]","[Legality, constitutionality and jurisprudence...",mfc
1,975,Immigration1.0-20547,Press One for English,immigration,The immigration debate in Congress has hit sev...,"{'annotator3': [4, 11, 13], 'annotator6': [3, ...","[3, 11, 4, 13]","[Morality, Cultural identity, Fairness and equ...",mfc
2,976,Immigration1.0-17350,Prosecutors Say Defendant in Immigrant Smuggli...,immigration,The Chinese businesswoman called Sister Ping w...,"{'annotator1': [1, 7, 9, 11], 'annotator10': [...","[1, 11, 9, 7]","[Economic, Cultural identity, Health and safet...",mfc
3,977,Immigration1.0-19222,Protests Go On In Several Cities As Panel Acts,immigration,Tens of thousands of immigrants here and in se...,"{'annotator8': [5, 7, 8, 11, 12], 'annotator9'...","[5, 6, 7, 8, 11, 12, 13]","[Legality, constitutionality and jurisprudence...",mfc
4,978,Immigration1.0-18843,Refining The Tests That Confer Citizenship,immigration,Where does Father Christmas come from? How old...,"{'annotator3': [1, 5, 6], 'annotator7': [6, 15]}","[1, 5, 6, 15]","[Economic, Legality, constitutionality and jur...",mfc


In [130]:
# Load SemEval subtask-2 from Postgres and map labels
conn = psycopg2.connect(
    dbname=os.getenv('DB_NAME'),
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD'),
    host=os.getenv('DB_HOST'),
    port=os.getenv('DB_PORT'),
)
cur = conn.cursor()
cur.execute("""
    SELECT article_id, title, text, frames_mfc, frames_raw, split, source
    FROM semeval_subtask2
    WHERE text IS NOT NULL
      AND (frames_mfc IS NOT NULL OR frames_raw IS NOT NULL)
""")
rows = cur.fetchall()
cur.close()
conn.close()

semeval_records = []
for article_id, title, text, frames_mfc, frames_raw, split, source in rows:
    # Prefer pre-mapped MFC labels if present
    labels = None
    if isinstance(frames_mfc, list) and len(frames_mfc) > 0:
        labels = frames_mfc
    elif isinstance(frames_raw, list) and len(frames_raw) > 0:
        labels = [SEMEVAL_TO_LABEL.get(f) for f in frames_raw]
        labels = [l for l in labels if l]

    if not labels:
        continue
    if not text or not str(text).strip():
        continue

    # Ensure labels align to official set
    labels = [l for l in labels if l in OFFICIAL_LABELS]
    if not labels:
        continue

    semeval_records.append({
        'source': 'semeval',
        'article_id': article_id,
        'title': title,
        'topic': None,
        'text': text,
        'labels': sorted(set(labels)),
        'split': split,
        'orig_source': source,
    })

semeval_df = pd.DataFrame(semeval_records)
print('SemEval rows:', len(semeval_df))
print(semeval_df['split'].value_counts(dropna=False))
semeval_df.head()


SemEval rows: 516
split
train    433
dev       83
Name: count, dtype: int64


,source,article_id,title,topic,text,labels,split,orig_source
0,semeval,111111111,Next plague outbreak in Madagascar could be 's...,None,Geneva - The World Health Organisation chief o...,"[Health and safety, Quality of life]",train,semeval_2023_task3_subtask2_en
1,semeval,111111112,US bloggers banned from entering UK,None,Two prominent US bloggers have been banned fro...,"[Crime and punishment, External regulation and...",train,semeval_2023_task3_subtask2_en
2,semeval,111111113,Kate Steinle's death at the hands of a Mexican...,None,The surprise acquittal of Jose Ines Garcia Zar...,"[Crime and punishment, Legality, constitutiona...",train,semeval_2023_task3_subtask2_en
3,semeval,111111114,U.S. judge frees Indonesian immigrant held by ...,None,A U.S. judge on Wednesday ordered the release ...,"[Crime and punishment, Fairness and equality, ...",train,semeval_2023_task3_subtask2_en
4,semeval,111111115,Here are all the sexual misconduct accusations...,None,Sen. Al Franken announced his resignationon th...,"[Crime and punishment, Legality, constitutiona...",train,semeval_2023_task3_subtask2_en


In [131]:
semeval_df.columns, mfc.columns

(Index(['source', 'article_id', 'title', 'topic', 'text', 'labels', 'split',
        'orig_source'],
       dtype='object'),
 Index(['id', 'article_id', 'title', 'topic', 'text_cleaned',
        'frame_annotations', 'present_frames_keys', 'present_frames', 'source'],
       dtype='object'))

In [132]:
# Step 1: Align columns and merge gold datasets

# MFC columns needed: source, article_id, title, text, labels, topic
mfc_aligned = mfc[['source', 'article_id', 'title', 'text_cleaned', 'present_frames', 'topic']].copy()
mfc_aligned = mfc_aligned.rename(columns={
    'text_cleaned': 'text',
    'present_frames': 'labels'
})

# SemEval columns needed: source, article_id, title, text, labels, topic (None for now)
semeval_aligned = semeval_df[['source', 'article_id', 'title', 'text', 'labels', 'topic']].copy()

print("MFC shape:", mfc_aligned.shape)
print("SemEval shape:", semeval_aligned.shape)
print("\nMFC columns:", mfc_aligned.columns.tolist())
print("SemEval columns:", semeval_aligned.columns.tolist())

MFC shape: (2224, 6)
SemEval shape: (516, 6)

MFC columns: ['source', 'article_id', 'title', 'text', 'labels', 'topic']
SemEval columns: ['source', 'article_id', 'title', 'text', 'labels', 'topic']


In [133]:
# Step 2: Stack the DataFrames
gold_combined = pd.concat([mfc_aligned, semeval_aligned], ignore_index=True)
print(f"Combined gold dataset: {len(gold_combined)} articles")
print(f"\nSource distribution:")
print(gold_combined['source'].value_counts())
print(f"\nTopic distribution (MFC has topics, SemEval is None):")
print(gold_combined['topic'].value_counts(dropna=False))

Combined gold dataset: 2740 articles

Source distribution:
source
mfc        2224
semeval     516
Name: count, dtype: int64

Topic distribution (MFC has topics, SemEval is None):
topic
immigration    1128
samesex         620
None            516
smoking         476
Name: count, dtype: int64


In [134]:
# Step 3: Convert labels to integer form using OFFICIAL_LABELS index
# Create mapping from label name to index
LABEL_TO_IDX = {label: idx for idx, label in enumerate(OFFICIAL_LABELS)}
print("Label to index mapping:")
for label, idx in LABEL_TO_IDX.items():
    print(f"  {idx}: {label}")

def labels_to_indices(label_list):
    """Convert list of label strings to sorted list of indices."""
    if not isinstance(label_list, list):
        return []
    indices = [LABEL_TO_IDX[l] for l in label_list if l in LABEL_TO_IDX]
    return sorted(indices)

gold_combined['labels_idx'] = gold_combined['labels'].apply(labels_to_indices)

# Verify conversion
print(f"\nSample label conversions:")
for i in range(min(3, len(gold_combined))):
    print(f"  {gold_combined.iloc[i]['labels'][:2]}... -> {gold_combined.iloc[i]['labels_idx'][:2]}...")

Label to index mapping:
  0: Economic
  1: Capacity and resources
  2: Morality
  3: Fairness and equality
  4: Legality, constitutionality and jurisprudence
  5: Policy prescription and evaluation
  6: Crime and punishment
  7: Security and defense
  8: Health and safety
  9: Quality of life
  10: Cultural identity
  11: Public opinion
  12: Political
  13: External regulation and reputation
  14: Other

Sample label conversions:
  ['Legality, constitutionality and jurisprudence', 'Policy prescription and evaluation']... -> [4, 5]...
  ['Morality', 'Cultural identity']... -> [2, 3]...
  ['Economic', 'Cultural identity']... -> [0, 6]...


In [135]:
# Step 4: Run topic classifier for articles without topics
# Import the topic classifier utilities
import sys
project_root = Path('.').resolve().parent
sys.path.insert(0, str(project_root / 'scripts' / 'utils'))
from topic_classifier_utils import TopicClassifier, TOPIC_LABELS

# MFC topic -> 19-topic mapping (use classifier's expected format)
MFC_TOPIC_MAP = {
    'immigration': 'Immigration',
    'smoking': 'Health',
    'samesex': 'Social Issues',  # Primary mapping for same-sex marriage
}

# Initialize topic classifier
classifier = TopicClassifier(model_path=str(project_root / 'notebooks' / 'saved_models' / 'final_topic_classifier'))

# Apply MFC topic mappings for MFC articles
mfc_idxs = gold_combined[gold_combined['source'] == 'mfc'].index
gold_combined.loc[mfc_idxs, 'pred_topic'] = gold_combined.loc[mfc_idxs, 'topic'].map(MFC_TOPIC_MAP)

print(f"MFC articles with pred_topic assigned: {gold_combined.loc[mfc_idxs, 'pred_topic'].notna().sum()}")
print(f"SemEval articles needing classification: {len(gold_combined[gold_combined['source'] == 'semeval'])}")

Loading topic classifier from C:\Users\rhrou\Documents\DS_Projects_Local\frame-delta\notebooks\saved_models\final_topic_classifier...
Using device: cuda
Loaded model with 19 topic classes
MFC articles with pred_topic assigned: 2224
SemEval articles needing classification: 516


In [136]:
# Run classifier on SemEval articles

sem_eval_idxs = gold_combined[gold_combined.source == 'semeval'].index
texts_to_classify = gold_combined.loc[sem_eval_idxs]['text'].to_list()
predicted_topics = classifier.predict_batch(texts_to_classify, batch_size=16)
gold_combined.loc[sem_eval_idxs, 'pred_topic'] = predicted_topics
    
print(f"\nTopic distribution after classification:")
print(gold_combined['pred_topic'].value_counts())

Predicting topics: 100%|██████████| 33/33 [00:03<00:00, 10.06it/s]


Topic distribution after classification:
pred_topic
Immigration             1160
Social Issues            624
Health                   500
Politics                 219
Legal                     82
Crime & Safety            66
Lifestyle & Culture       36
War & Conflict            19
Science & Technology      12
Education                 10
Media                      4
Entertainment              3
Disaster & Accidents       2
Other/Unknown              1
Environment & Nature       1
Business & Economy         1
Name: count, dtype: int64


In [137]:
# Step 5: Format text with topic prefix
# Format must EXACTLY match silver training: TOPIC:{topic}\n{title}\n{text}
# NOTE: No space after colon - this is critical for tokenization alignment

def format_text_with_topic(row):
    """Format article text with topic prefix matching silver training format."""
    topic = row['pred_topic']
    title = row['title'] if pd.notna(row['title']) else ''
    text = row['text'] if pd.notna(row['text']) else ''
    
    # Match the EXACT format used in silver training (no space after colon)
    return f"TOPIC:{topic}\n{title}\n{text}"

gold_combined['formatted_text'] = gold_combined.apply(format_text_with_topic, axis=1)

# Verify format matches silver training
print("Sample formatted text (first 300 chars):")
print("-" * 50)
print(gold_combined.iloc[0]['formatted_text'][:300])
print("-" * 50)
print("\nSemEval sample:")
print("-" * 50)
semeval_sample = gold_combined[gold_combined['source'] == 'semeval'].iloc[0]
print(semeval_sample['formatted_text'][:300])
print("-" * 50)

# Verify format structure
sample = gold_combined.iloc[0]['formatted_text']
assert sample.startswith("TOPIC:"), "ERROR: Must start with TOPIC:"
assert not sample.startswith("TOPIC: "), "ERROR: Should NOT have space after colon"
print("\nFormat verification PASSED - matches silver training structure")

Sample formatted text (first 300 chars):
--------------------------------------------------
TOPIC:Immigration
THE FINE PRINT: A close look at the immigration bill.; Change at the Border Could Pinch the Arts
By nearly all accounts, Karen Zacarias, a 26-year-old playwright from Mexico, seems destined for success.
Since writing her first prize-winning dramatic comedy five years ago as a stude
--------------------------------------------------

SemEval sample:
--------------------------------------------------
TOPIC:Health
Next plague outbreak in Madagascar could be 'stronger': WHO
Geneva - The World Health Organisation chief on Wednesday said a deadly plague epidemic appeared to have been brought under control in Madagascar, but warned the next outbreak would likely be stronger.

"The next transmission c
--------------------------------------------------

Format verification PASSED - matches silver training structure


In [138]:
# Step 6: Prepare final DataFrame for database
# Select columns for gold_train_data table
gold_final = gold_combined[[
    'source',
    'article_id', 
    'title',
    'text',
    'pred_topic',
    'formatted_text',
    'labels',       # String label names
    'labels_idx',   # Integer indices
]].copy()

# Rename for clarity
gold_final = gold_final.rename(columns={
    'pred_topic': 'topic',
})

# Convert label lists to JSON strings for PostgreSQL storage
gold_final['labels_json'] = gold_final['labels'].apply(json.dumps)
gold_final['labels_idx_json'] = gold_final['labels_idx'].apply(json.dumps)

print("Final DataFrame shape:", gold_final.shape)
print("\nColumns:", gold_final.columns.tolist())
print("\nSample row:")
gold_final.iloc[0]

Final DataFrame shape: (2740, 10)

Columns: ['source', 'article_id', 'title', 'text', 'topic', 'formatted_text', 'labels', 'labels_idx', 'labels_json', 'labels_idx_json']

Sample row:


source                                                           mfc
article_id                                       Immigration1.0-1371
title              THE FINE PRINT: A close look at the immigratio...
text               By nearly all accounts, Karen Zacarias, a 26-y...
topic                                                    Immigration
formatted_text     TOPIC:Immigration\nTHE FINE PRINT: A close loo...
labels             [Legality, constitutionality and jurisprudence...
labels_idx                                         [4, 5, 9, 10, 11]
labels_json        ["Legality, constitutionality and jurisprudenc...
labels_idx_json                                    [4, 5, 9, 10, 11]
Name: 0, dtype: object

In [139]:
# Step 7: Create and populate gold_train_data table in database
from sqlalchemy import create_engine, text

# Create SQLAlchemy engine
db_url = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(db_url)

# Create table schema
create_table_sql = """
DROP TABLE IF EXISTS gold_train_data;
CREATE TABLE gold_train_data (
    id SERIAL PRIMARY KEY,
    source VARCHAR(50) NOT NULL,
    article_id VARCHAR(100) NOT NULL,
    title TEXT,
    text TEXT NOT NULL,
    topic VARCHAR(100) NOT NULL,
    formatted_text TEXT NOT NULL,
    labels_json TEXT NOT NULL,
    labels_idx_json TEXT NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
CREATE INDEX idx_gold_train_source ON gold_train_data(source);
CREATE INDEX idx_gold_train_topic ON gold_train_data(topic);
"""

with engine.connect() as conn:
    conn.execute(text(create_table_sql))
    conn.commit()
print("Table gold_train_data created successfully")

Table gold_train_data created successfully


In [140]:
# Insert data into gold_train_data table
# Prepare DataFrame for insertion (select only DB columns, exclude list columns)
df_to_insert = gold_final[[
    'source', 'article_id', 'title', 'text', 'topic', 
    'formatted_text', 'labels_json', 'labels_idx_json'
]].copy()

# Upload to database
df_to_insert.to_sql(
    'gold_train_data',
    engine,
    if_exists='append',  # Table already created above
    index=False,
    method='multi',
    chunksize=100
)

print(f"Uploaded {len(df_to_insert)} rows to gold_train_data table")

Uploaded 2740 rows to gold_train_data table


In [143]:
# Step 8: Verify upload and print summary
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM gold_train_data"))
    total = result.fetchone()[0]
    
    result = conn.execute(text("""
        SELECT source, COUNT(*) as count 
        FROM gold_train_data 
        GROUP BY source
    """))
    source_counts = result.fetchall()
    
    result = conn.execute(text("""
        SELECT topic, COUNT(*) as count 
        FROM gold_train_data 
        GROUP BY topic 
        ORDER BY count DESC
    """))
    topic_counts = result.fetchall()

print("=" * 50)
print("GOLD TRAIN DATA ASSEMBLY COMPLETE")
print("=" * 50)
print(f"\nTotal articles: {total}")
print(f"\nBy source:")
for source, count in source_counts:
    print(f"  {source}: {count}")
print(f"\nBy topic:")
for topic, count in topic_counts:
    print(f"  {topic}: {count}")
print("\n" + "=" * 50)
print("Next step: Create gold_training_run.ipynb for experiments")
print("=" * 50)

GOLD TRAIN DATA ASSEMBLY COMPLETE

Total articles: 2740

By source:
  mfc: 2224
  semeval: 516

By topic:
  Immigration: 1160
  Social Issues: 624
  Health: 500
  Politics: 219
  Legal: 82
  Crime & Safety: 66
  Lifestyle & Culture: 36
  War & Conflict: 19
  Science & Technology: 12
  Education: 10
  Media: 4
  Entertainment: 3
  Disaster & Accidents: 2
  Business & Economy: 1
  Other/Unknown: 1
  Environment & Nature: 1

Next step: Create gold_training_run.ipynb for experiments


In [142]:
# Step 9: Analyze label distribution for training planning
from collections import Counter

# Count all labels
all_labels = []
for labels in gold_final['labels']:
    all_labels.extend(labels)

label_counts = Counter(all_labels)
print("Label distribution in gold dataset:")
print("-" * 50)
for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    pct = count / len(gold_final) * 100
    print(f"{label:45s}: {count:4d} ({pct:5.1f}%)")

print(f"\nTotal articles: {len(gold_final)}")
print(f"Avg labels per article: {sum(label_counts.values()) / len(gold_final):.2f}")

Label distribution in gold dataset:
--------------------------------------------------
Legality, constitutionality and jurisprudence: 1600 ( 58.4%)
Political                                    : 1373 ( 50.1%)
Policy prescription and evaluation           : 1127 ( 41.1%)
Crime and punishment                         :  879 ( 32.1%)
Economic                                     :  870 ( 31.8%)
Quality of life                              :  825 ( 30.1%)
Cultural identity                            :  737 ( 26.9%)
Public opinion                               :  688 ( 25.1%)
Health and safety                            :  664 ( 24.2%)
Fairness and equality                        :  599 ( 21.9%)
Morality                                     :  590 ( 21.5%)
Security and defense                         :  451 ( 16.5%)
External regulation and reputation           :  364 ( 13.3%)
Other                                        :  304 ( 11.1%)
Capacity and resources                       :  276 ( 10.1%